# Inference Demo

Shows the full pipeline on sample images: features, CNN score, issue detection, 
heatmap, and the explainability output.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import cv2, base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from config import DATA_DIR, MODELS_DIR
from ml.inference import QualityAnalyzer

DATA_GEN = DATA_DIR / "generated"
analyzer = QualityAnalyzer(MODELS_DIR)
print(f"Model: {analyzer.model_type}")
print(f"Classifiers: {list(analyzer.classifiers.keys())}")

## Pick diverse test samples

In [ ]:
test_meta = pd.read_csv(DATA_GEN / "test" / "metadata.csv")
img_dir = DATA_GEN / "test" / "images"

# one of each type
sample_rows = []
for dt in ["GOOD", "BLUR", "NOISE", "UNDEREXPOSURE", "OVEREXPOSURE", 
           "LOW_CONTRAST", "JPEG_CORRUPTION", "SEVERE_DEGRADATION"]:
    sub = test_meta[test_meta['degradation_type'] == dt]
    if len(sub):
        sample_rows.append(sub.iloc[len(sub)//2])
print(f"Testing {len(sample_rows)} images")

## Run analysis on each

In [ ]:
results = []
for row in sample_rows:
    img_bytes = open(str(img_dir / row['filename']), 'rb').read()
    r = analyzer.analyze(img_bytes)
    results.append(r)
    
    # display
    img = cv2.cvtColor(cv2.imread(str(img_dir / row['filename'])), cv2.COLOR_BGR2RGB)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    axes[0].imshow(img)
    axes[0].set_title(f"{row['degradation_type']} (sev {row['severity']})\nGT: {row['quality_score']:.0f}")
    axes[0].axis('off')
    
    if r['heatmap']:
        hm = cv2.imdecode(np.frombuffer(base64.b64decode(r['heatmap'].split(',')[1]), np.uint8), cv2.IMREAD_COLOR)
        axes[1].imshow(cv2.cvtColor(hm, cv2.COLOR_BGR2RGB))
        axes[1].set_title("CNN attention heatmap")
    axes[1].axis('off')
    
    plt.suptitle(f"Score: {r['quality_score']:.1f} — {r['quality_label']} ({r['confidence']:.0%})", fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # print issues
    if r['issues']:
        for iss in r['issues']:
            evidence = ', '.join(f"{e['feature']}={e['value']}" for e in iss.get('evidence', []))
            print(f"  {iss['type']}: {iss['severity']} ({iss['confidence']:.0%}) — {evidence}")
    else:
        print("  No issues detected")
    print()

## Summary table

In [ ]:
summary = []
for row, r in zip(sample_rows, results):
    summary.append({
        "Type": row['degradation_type'],
        "Severity": row['severity'],
        "GT Score": row['quality_score'],
        "Predicted": r['quality_score'],
        "Label": r['quality_label'],
        "Issues": len(r['issues']),
        "Conf": f"{r['confidence']:.0%}",
    })
print(pd.DataFrame(summary).to_string(index=False))

## How explainability works

Two levels:

1. **Feature evidence** — each issue detection shows which features drove the decision
   (e.g. blur detected because `laplacian_variance=12` is very low)

2. **Grad-CAM heatmap** — shows where the CNN focused spatially, highlighting
   the regions that most influenced the quality score

This means every prediction can be inspected and understood, which is
important for building trust in automated quality assessment.

In [ ]:
print("To run the full server: uvicorn backend.app:app --reload")